## Pod security context — user, perms & capabilities

By default containers run as **root** with a broad set of Linux capabilities — both dangerous. The **security context** is your hardening toolkit, at pod level and container level (container overrides pod):

```yaml
spec:
  securityContext:                 # pod-level
    runAsNonRoot: true
    runAsUser: 1000
    fsGroup: 2000
    seccompProfile: { type: RuntimeDefault }
  containers:
    - name: app
      securityContext:             # container-level
        allowPrivilegeEscalation: false
        readOnlyRootFilesystem: true
        capabilities: { drop: ["ALL"], add: ["NET_BIND_SERVICE"] }
```

Field by field:

- **`runAsUser` / `runAsGroup`** — the UID/GID the process runs as (numbers, not names).
- **`runAsNonRoot: true`** — the container **fails to start** if it would run as UID 0. The cheapest safety net.
- **`fsGroup`** — the GID applied to mounted volumes, so a non-root container can write.
- **`allowPrivilegeEscalation: false`** — no `setuid` tricks; combined with non-root, no path to root.
- **`readOnlyRootFilesystem: true`** — `/` read-only (writable paths need an `emptyDir`); kills "drop a binary in `/tmp`" exploits.
- **`capabilities.drop: ["ALL"]`** then `add` the few you need.
- **`seccompProfile: RuntimeDefault`** — the runtime's syscall filter.

**`privileged: true`** turns *all* of this off — full root on the node. **Only node-managing DaemonSets** (CNI, CSI) should set it. Common trip-up: `runAsNonRoot: true` on a root image fails — set `runAsUser` or build with `USER 1000` (distroless/Chainguard ship non-root). On our map these are the **runAsNonRoot** and **read-only fs** chips in the Security box — the per-Pod hardening the next section enforces cluster-wide.